# CyberSecRAG — Live Retrieval Demo

This notebook demonstrates the live retrieval and answer generation workflow of CyberSecRAG, a cybersecurity threat intelligence system built for grounded question answering over CVE records. Retrieval-Augmented Generation (RAG) first finds the most relevant vulnerability records and then asks the language model to answer using only that retrieved evidence.

In this project, the student built a complete pipeline that processes CVE data, generates embeddings, indexes the knowledge base with FAISS, retrieves the most relevant vulnerabilities for a user query, and produces a factual response backed by real CVE entries.

## Setup — Load the Pipeline

In [13]:
# Move the notebook process to the project root so all file paths behave the same way as the CLI.
# Reload project modules so the notebook picks up fresh code changes even after earlier failed runs.
import sys, os, importlib
project_root = os.path.abspath("..")
os.chdir(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(".env")

import src.retriever as retriever_module
import src.generator as generator_module
import src.pipeline as pipeline_module

importlib.reload(retriever_module)
importlib.reload(generator_module)
importlib.reload(pipeline_module)

from src.retriever import search
from src.generator import generate
from src.pipeline import query as run_query

print("Pipeline loaded successfully.")

Pipeline loaded successfully.


## Helper — Display Function

In [14]:
# Print the query, answer, and retrieved CVEs in a clean plain-text format for live demos.
# Use only print() so the output stays simple and predictable during the viva.
def show_results(query_text, result):
    line = "-" * 90
    print(line)
    print(f"QUERY: {query_text}")
    print(line)
    print("ANSWER:")
    print(result.get("answer", "No answer returned."))
    print(line)
    print("RETRIEVED CVES:")

    cves = result.get("retrieved_cves", [])
    if not cves:
        print("No CVE results were retrieved.")
        print(line)
        return

    for index, cve in enumerate(cves, start=1):
        description = str(cve.get("description", "Not available"))
        snippet = description[:200] + ("..." if len(description) > 200 else "")
        score = float(cve.get("score", 0.0)) * 100
        severity = cve.get("severity") or "Not available"
        cvss = cve.get("cvss") if cve.get("cvss") is not None else "N/A"

        print(f"{index}. {cve.get('cve_id', 'Unknown CVE')}")
        print(f"   Severity: {severity}")
        print(f"   CVSS: {cvss}")
        print(f"   Score: {score:.1f}%")
        print(f"   Description: {snippet}")
        print(line)

## Query 1 — Remote Code Execution

Remote code execution vulnerabilities are especially dangerous because they can let an attacker run arbitrary code on a target system from a remote location.

In [15]:
# Run the full RAG pipeline for a remote code execution query.
# Show the final answer together with the supporting retrieved CVEs.
result = run_query("critical remote code execution vulnerabilities in 2024")
show_results("critical remote code execution vulnerabilities in 2024", result)

------------------------------------------------------------------------------------------
QUERY: critical remote code execution vulnerabilities in 2024
------------------------------------------------------------------------------------------
ANSWER:
Based on the provided CVE records, the following critical remote code execution vulnerabilities were identified in 2024:

1. CVE-2024-2195 is a critical Remote Code Execution (RCE) vulnerability in the aimhubio/aim project, specifically within the `/api/runs/search/run/` endpoint, affecting versions >= 3.0.0, due to improper restriction of user access to the `RunView` object.
2. CVE-2024-42506 is a critical Command Injection vulnerability in the underlying CLI service of Hewlett Packard Enterprise (HPE) Aruba OS, allowing unauthenticated remote code execution by sending specially crafted packets destined to the PAPI UDP port (8211).
3. CVE-2024-1015 is a critical Remote Command Execution vulnerability in SE-elektronic GmbH E-DDC3.3 affect

## Query 2 — SQL Injection

SQL injection vulnerabilities happen when attacker-controlled input is interpreted as part of a database query, allowing unauthorized access or manipulation.

In [16]:
# Run the full RAG pipeline for an SQL injection query.
# Print the answer and the evidence returned by retrieval.
result = run_query("SQL injection vulnerabilities in web applications")
show_results("SQL injection vulnerabilities in web applications", result)

------------------------------------------------------------------------------------------
QUERY: SQL injection vulnerabilities in web applications
------------------------------------------------------------------------------------------
ANSWER:
The following CVEs match the query for SQL injection vulnerabilities in web applications:

1. CVE-2023-22378 is a HIGH severity vulnerability with a CVSS score of 8.8, affecting Nozomi Networks Guardian and CMC. It is a CWE-89 SQL injection vulnerability that allows an authenticated attacker to execute arbitrary SQL statements on the DBMS used by the web application.

2. CVE-2023-0895 is a HIGH severity vulnerability with a CVSS score of 7.2, affecting the WP Coder – add custom html, css and js code plugin for WordPress. It is a CWE-89 SQL injection vulnerability that allows authenticated attackers with administrative privileges to extract sensitive information from the database.

3. CVE-2023-20010 is a HIGH severity vulnerability with a CVSS 

## Query 3 — Authentication Bypass

Authentication bypass vulnerabilities allow an attacker to gain access to a system or protected function without passing the normal identity checks.

In [17]:
# Run the full RAG pipeline for an authentication bypass query.
# Print the answer and the retrieved vulnerability records.
result = run_query("authentication bypass vulnerabilities")
show_results("authentication bypass vulnerabilities", result)

------------------------------------------------------------------------------------------
QUERY: authentication bypass vulnerabilities
------------------------------------------------------------------------------------------
ANSWER:
Based on the provided CVE records, the following vulnerabilities match the query for authentication bypass vulnerabilities.

1. CVE-2023-38123 is a HIGH severity vulnerability with a CVSS score of 7.5. It affects Inductive Automation Ignition and is caused by a missing authentication for critical function, allowing remote attackers to bypass authentication. User interaction is required to exploit this vulnerability.

2. CVE-2023-32152 is a MEDIUM severity vulnerability with a CVSS score of 6.5. It affects D-Link DIR-2640 and is caused by an incorrect implementation of an authentication algorithm, allowing network-adjacent attackers to bypass authentication without requiring proper credentials.

3. CVE-2023-41183 is a HIGH severity vulnerability with a CVS

## Query 4 — Buffer Overflow

In [18]:
# Run the full RAG pipeline for a buffer overflow and memory corruption query.
# Print the answer and the supporting CVEs in a readable format.
result = run_query("buffer overflow memory corruption vulnerabilities")
show_results("buffer overflow memory corruption vulnerabilities", result)

------------------------------------------------------------------------------------------
QUERY: buffer overflow memory corruption vulnerabilities
------------------------------------------------------------------------------------------
ANSWER:
In the realm of cybersecurity, buffer overflow vulnerabilities pose a significant threat to system integrity. A buffer overflow occurs when more data is written to a buffer than it is designed to hold, potentially causing memory corruption. Based on the provided CVE records, the following vulnerabilities match the query for buffer overflow memory corruption vulnerabilities.

1. CVE-2024-45620 (2024, LOW, CVSS: 3.9) affects Red Hat Enterprise Linux 7, 8, 9, and 10, and is a classic buffer overflow vulnerability (CWE-120) due to buffer copy without checking size of input, allowing an attacker to potentially cause memory corruption.

These CVEs demonstrate the importance of addressing buffer overflow vulnerabilities to prevent memory corruption a

## Query 5 — Apache / Open Source Software

In [19]:
# Run the full RAG pipeline for a query focused on Apache software in 2024.
# Print the final answer and the retrieved CVE evidence for the demo audience.
result = run_query("critical vulnerabilities in Apache software 2024")
show_results("critical vulnerabilities in Apache software 2024", result)

------------------------------------------------------------------------------------------
QUERY: critical vulnerabilities in Apache software 2024
------------------------------------------------------------------------------------------
ANSWER:
No relevant CVEs were found for your query.
------------------------------------------------------------------------------------------
RETRIEVED CVES:
No CVE results were retrieved.
------------------------------------------------------------------------------------------


## Performance Measurement

In [20]:
# Measure retrieval and generation separately so the viva can discuss system speed clearly.
# Report all timings in milliseconds because that is easy to interpret in a live demo.
import time

test_query = "remote code execution in network devices"

# Measure retrieval only
start = time.time()
cves = search(test_query, top_k=5)
retrieval_time = (time.time() - start) * 1000

# Measure generation only
start = time.time()
answer = generate(test_query, cves)
generation_time = (time.time() - start) * 1000

print(f"Retrieval latency:  {retrieval_time:.1f} ms")
print(f"Generation latency: {generation_time:.1f} ms")
print(f"Total latency:      {retrieval_time + generation_time:.1f} ms")

Retrieval latency:  31.0 ms
Generation latency: 1953.8 ms
Total latency:      1984.8 ms


## Key Takeaways

- RAG goes beyond keyword search by retrieving semantically relevant CVE records, which means the system can still find strong matches even when the user does not use the exact wording from the dataset.
- The knowledge base contains tens of thousands of CVE records across 2023 and 2024, giving the system enough coverage to answer a wide range of cybersecurity questions from a grounded source.
- The embedding model used is `all-MiniLM-L6-v2`, which is a strong fit for this project because it is lightweight, fast to run locally, and produces 384-dimensional vectors that work well for semantic similarity search.
- The retrieval latency shows that vector search is fast enough for interactive use, making the system suitable for a live demo where relevant CVEs can be found in well under a second before answer generation begins.